In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modelling 
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

# ==========================================
# 1. LOAD DATA & INITIAL CHECKS
# ==========================================
file_path = 'data/stud.csv'
if not os.path.exists(file_path):
    raise FileNotFoundError(f"Missing dataset! Please check that '{file_path}' exists.")

df = pd.read_csv(file_path)

# Quick validation check to prevent the 'Less than two samples' NaN issue
if len(df) <= 5:
    print(f"⚠️ WARNING: Your stud.csv only has {len(df)} rows. Model metrics might show NaN.")
else:
    print(f"✅ Success: Loaded {len(df)} rows of student data.")

# ==========================================
# 2. PREPARE X AND Y VARIABLES
# ==========================================
# We are predicting 'math_score' (Dependent Variable)
X = df.drop(columns=['math_score'], axis=1)
y = df['math_score']

# ==========================================
# 3. CREATE PIPELINE PIPELINE (COLUMN TRANSFORMER)
# ==========================================
num_features = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(include="object").columns

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features),        
    ]
)

# Fit and transform the feature matrix X
X = preprocessor.fit_transform(X)
print(f"Shape of feature matrix X after transformation: {X.shape}")

# ==========================================
# 4. TRAIN TEST SPLIT
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")

# ==========================================
# 5. EVALUATION FUNCTION
# ==========================================
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

# ==========================================
# 6. MODEL TRAINING LOOP & REPORTING
# ==========================================
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(), 
    "AdaBoost Regressor": AdaBoostRegressor()
}

model_list = []
r2_list = []

for name, model in models.items():
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    print(f"\n👉 {name}")
    model_list.append(name)
    r2_list.append(model_test_r2)
    
    print('Model performance for Training set:')
    print(f"- Root Mean Squared Error: {model_train_rmse:.4f}")
    print(f"- Mean Absolute Error: {model_train_mae:.4f}")
    print(f"- R2 Score: {model_train_r2:.4f}")
    print('----------------------------------')
    print('Model performance for Testing set:')
    print(f"- Root Mean Squared Error: {model_test_rmse:.4f}")
    print(f"- Mean Absolute Error: {model_test_mae:.4f}")
    print(f"- R2 Score: {model_test_r2:.4f}")

# ==========================================
# 7. FINAL RESULTS TABLE
# ==========================================
print("\n" + "="*40)
print("       FINAL PERFORMANCE REPORT")
print("="*40)
report_df = pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score'])
report_df = report_df.sort_values(by=["R2_Score"], ascending=False).reset_index(drop=True)
print(report_df)